## 2.2 文本预处理 - 词表 Vocabulary 与 token 到数字的映射

#### 1、为什么这一小节非常关键

##### 1.1 这是文本“数值化”的第一步
我们上一小节已经学过：

- 原始文本不能直接输入 RNN
- 文本要先切分成 token 序列
- token 可以是字符、单词、子词等基本单位

但是即使已经完成了分词，模型依然不能直接处理这些 token。

例如一句话分词后变成：

`I / love / AI`

这仍然只是字符串序列，不是数值。

而神经网络能处理的，必须是数字张量。

所以接下来就必须解决一个核心问题：

如何把 token 变成数字？

这就是这一小节的主题。

##### 1.2 这一小节连接了“文本世界”和“模型世界”
从人的角度看：

`I / love / AI`

已经很清楚了。

但从模型的角度看，它需要的是类似这样的输入：

`2 / 5 / 9`

也就是说，我们必须建立一套规则，把每个 token 对应到一个唯一的数字编号。

这个过程本质上就是在做一座桥梁：

人类可读的 token  
$\rightarrow$  
机器可处理的数字 id

所以这一节是整个 NLP 处理中非常关键的过渡环节。

##### 1.3 没有这一步，后面的 Embedding 和 RNN 都无法进行
后面我们会学习：

- Embedding 层
- Padding
- RNN 输入张量
- batch 处理

但这些内容都有一个共同前提：

token 必须先变成数字 id

因为 Embedding 层查找的不是字符串，而是索引。

RNN 接收的也不是单词，而是经过进一步处理后的数值表示。

所以这节内容虽然还没有真正进入模型结构，但它是后面一切操作的基础。


#### 2、为什么 token 不能直接送进模型

##### 2.1 token 本质上仍然是字符串
例如：

- `I`
- `love`
- `AI`

这些 token 对人类来说是有意义的，但对模型来说，它们仍然只是文本符号。

模型不能直接拿字符串做矩阵运算。

所以哪怕已经完成了分词，仍然不够。

##### 2.2 神经网络处理的是数值，不是文字
神经网络本质上做的是：

- 加法
- 乘法
- 矩阵运算
- 梯度更新

这些运算都只能建立在数值基础上。

例如我们可以计算：

$3 \times 5$

$2 + 7$

但我们不能计算：

$\text{love} \times \text{AI}$

$\text{I} + \text{movie}$

所以 token 如果想进入模型，必须先被转成某种数字形式。

##### 2.3 文字进入模型前，一定要经过“编号化”
这里可以先建立一个非常重要的概念：

文本数值化，不是直接把单词变成“有含义的数学值”，而是先给每个 token 分配一个编号。

例如：

- `I → 1`
- `love → 2`
- `AI → 3`

注意这里的 `1`、`2`、`3` 暂时只是编号，不代表大小关系，不代表语义强弱，也不代表 `love` 比 `I` 大。

它只是一个“身份标识”。

这一点非常重要，后面学习 Embedding 时你会更深刻地理解这一点。

#### 3、什么是词表 Vocabulary

##### 3.1 词表的本质
Vocabulary，简称 vocab，通常翻译为“词表”。

它的本质就是：

把数据集中会出现的 token 收集起来，并为每个 token 分配一个唯一编号。

例如我们有下面三句话：

- `I love AI`
- `AI is powerful`
- `I love coding`

先把它们切成 token，然后把所有出现过的 token 收集起来，可能得到：

- `I`
- `love`
- `AI`
- `is`
- `powerful`
- `coding`

接着为它们逐个编号：

- `I → 0`
- `love → 1`
- `AI → 2`
- `is → 3`
- `powerful → 4`
- `coding → 5`

这样就形成了一个最基础的 vocabulary。

##### 3.2 词表可以理解成“字典”
你可以把词表理解成一本字典，只不过这本字典不是解释单词意思，而是负责做“单词 $\rightarrow$ 编号”的映射。

例如：

- `I` 对应 `0`
- `love` 对应 `1`
- `AI` 对应 `2`

所以以后模型看到 `love` 这个 token 时，并不会直接处理 `love` 这个字符串，而是先把它查表变成数字 `1`。

##### 3.3 词表是整个文本数值化的核心工具
为什么一定要有词表？

因为模型不能直接认识 token，但它可以认识数字。

所以词表的作用就是：

把离散的语言符号，转成可索引的数字身份。

你可以把它理解为：

词表就是 NLP 世界里的“翻译器”，负责把文字翻译成模型能接收的编号系统。


#### 4、为什么必须建立词表

##### 4.1 因为模型需要统一的编号规则
如果没有词表，那么同一个 token 在不同地方就可能被随意编码，整个系统就会混乱。

例如今天你说：

`love → 2`

明天又说：

`love → 7`

那模型就无法稳定学习。

所以必须先建立一套统一规则：

每个 token 对应哪个 id，必须固定下来。

##### 4.2 因为后面所有样本都要共享同一套编码体系
假设我们有两个句子：

- `I love AI`
- `AI is great`

如果没有统一词表，那么这两个句子就没法放进同一个模型里共同训练。

只有先建立一个公共词表，例如：

- `I → 0`
- `love → 1`
- `AI → 2`
- `is → 3`
- `great → 4`

那么这两个句子才能分别变成：

`I love AI`  
$\rightarrow 0 \ 1 \ 2$

`AI is great`  
$\rightarrow 2 \ 3 \ 4$

这样模型才能在统一的编号空间中处理不同句子。

##### 4.3 因为后面的 Embedding 层本质上就是在查词表编号
后面我们会学到：

Embedding 层接收的是 token 的 id，然后查表取出对应向量。

这说明如果前面没有先把 token 编号，后面的 Embedding 根本无法工作。

所以词表不是可选项，而是文本输入模型前几乎必不可少的一步。

#### 5、词表是怎么建立出来的

##### 5.1 第一步：先拿到所有训练文本
建立词表时，首先要有一批文本数据。

例如：

- `I love AI`
- `AI is powerful`
- `I love coding`

##### 5.2 第二步：先分词，得到 token 序列
例如：

- `I / love / AI`
- `AI / is / powerful`
- `I / love / coding`

##### 5.3 第三步：统计所有出现过的 token
把所有 token 合并起来，再去重，可能得到：

- `I`
- `love`
- `AI`
- `is`
- `powerful`
- `coding`

这一步本质上是在回答：

训练数据里到底出现过哪些 token？

##### 5.4 第四步：给每个 token 分配唯一 id
例如：

- `I → 0`
- `love → 1`
- `AI → 2`
- `is → 3`
- `powerful → 4`
- `coding → 5`

这就形成了一个最基础的词表。

##### 5.5 实际项目中通常会有顺序规则
真实项目中，词表的编号方式不一定是随便排的，常见做法包括：

- 按出现顺序编号
- 按词频高低编号
- 先给特殊 token 固定编号，再给普通词编号

例如常见情况是：

- `<PAD> → 0`
- `<UNK> → 1`
- `AI → 2`
- `I → 3`
- `love → 4`

因为后面 padding 和未知词处理通常都依赖这些特殊 token，所以它们往往会优先占据前几个编号。


#### 6、什么叫 token 到 id 的映射

##### 6.1 映射的本质
所谓 token 到数字的映射，就是：

把一个 token 替换成它在词表中的编号。

例如词表是：

- `I → 2`
- `love → 3`
- `AI → 4`

那么句子：

`I / love / AI`

就会被映射为：

`2 / 3 / 4`

这个过程也常叫：

- numericalization
- encoding
- converting token to id

##### 6.2 这是文本从“符号序列”变成“数字序列”的关键一步
映射之前：

`I / love / AI`

映射之后：

`2 / 3 / 4`

前者是字符串序列，后者是数字序列。

只有变成数字序列之后，后面才能继续做：

- padding
- 转 tensor
- Embedding
- 输入 RNN

所以这一步就是文本数值化链条中最核心的一环。

##### 6.3 一个句子映射后，本质上就是一个整数序列
例如：

`This / movie / is / great`

假设词表中对应为：

- `This → 5`
- `movie → 8`
- `is → 3`
- `great → 10`

那么句子就会变成：

`5 / 8 / 3 / 10`

可以发现，到这里为止，文本已经开始变成“模型可处理的序列数据”了。

#### 7、为什么这些 id 只是编号，不是语义大小

##### 7.1 编号不代表数学大小关系
这一点必须非常明确。

例如：

- `good → 7`
- `bad → 8`
- `excellent → 9`

不能因为 `excellent` 的 id 比 `good` 大，就认为 `excellent` 更“强”。

这些数字只是标签，不是数值意义上的大小。

##### 7.2 编号只是为了唯一标识 token
id 的作用类似于身份证号或者学号。

例如：

- 张三是 `1001`
- 李四是 `1002`

你不会说 `1002` 就比 `1001` 更优秀。

同样地：

- `love → 3`
- `AI → 4`
- `movie → 5`

这里只是在做“身份区分”，不是在表达语义距离。

##### 7.3 这也是为什么后面还需要 Embedding
如果 id 本身就有语义，那么直接把这些数字送进去就行了。

但问题是：

id 只有编号功能，没有语义表达能力。

所以后面必须再经过 Embedding，把这些离散编号映射成稠密向量，模型才能学习到词与词之间的语义关系。

#### 8、什么是特殊 token，为什么必须有它们

在真实任务中，词表里通常不只有普通 token，还会专门加入一些“特殊 token”。

##### 8.1 补齐用的 token
不同句子的长度通常不一样，但 batch 训练时常常需要统一长度。

例如：

`I love AI`  
长度 `3`

`This movie is very good`  
长度 `5`

为了把它们放进同一个 batch，短句通常要补齐。

这时就会用到特殊 token。

例如规定长度为 `5`：

`I / love / AI / <PAD> / <PAD>`

然后再映射成数字。

所以特殊 token 的作用是：

用于占位，补齐序列长度。

##### 8.2 未知词 token
真实测试时，模型可能会遇到训练时没见过的新词。

例如训练词表里没有：

`ChatGPT`

这时就不能直接报错，否则模型没法运行。

通常做法是把它映射成一个统一的未知词标记：

`<UNK>`

也就是说：

任何不在词表中的 token  
$\rightarrow$  
统一映射到未知词的 id

所以未知词 token 的作用是：

处理词表外的新词，也就是 OOV 问题。

##### 8.3 序列开始标记
在某些任务里，会在句子开头加一个特殊 token，表示序列开始。

例如：

`<SOS>`

它的作用常见于：

- 序列生成
- 机器翻译
- decoder 输入

在基础文本分类任务中，不一定必须使用，但你需要知道这个概念。

##### 8.4 序列结束标记
类似地，有些任务会在句子末尾加上结束标记：

`<EOS>`

它表示句子到这里结束了。

在生成任务中尤其常见。


#### 9、词频和词表大小的现实问题

##### 9.1 真实语料中的 token 数量可能非常大
在真实任务中，文本数据远比课堂例子复杂。

如果你把所有出现过的 token 都放进词表，词表可能会变得非常大。

例如：

- 很多极少出现的词
- 拼写错误
- 特殊名字
- 数字变体
- 冗余符号

这些都可能让词表无限膨胀。

##### 9.2 词表太大，会带来很多问题
例如：

- 内存占用增大
- Embedding 参数量变大
- 稀有词太多，训练效果差
- 处理效率下降

所以真实项目中，通常不会无上限地保留所有 token。

##### 9.3 常见做法：只保留高频 token
例如只保留：

- 前 `10000` 个高频词
- 或出现次数至少 `2` 次、`5` 次以上的词

其余低频词统一映射到 `<UNK>`。

这样做的目的是：

在控制词表大小的同时，保留主要语义信息。

所以你要知道：

词表不是越大越好，而是要在表达能力和计算成本之间做平衡。